# MAINTiDOC — Semaine 2 : validation de la recherche avec BGE-M3 (Colab GPU)

**But.** Mesurer la recherche documentaire avec un vrai modèle d'embeddings (BAAI/bge-m3) et comparer au prototype `hashing` (non sémantique).

**Avant de lancer :** menu *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Tout exécuter*.

**Règles d'honnêteté**
- Le **premier passage** avec BGE-M3 est le résultat à rapporter, **avant tout réglage**. Il est enregistré dans `docs/eval_bge-m3_premier_passage.md`.
- Les seuils de `answer.py` (`MIN_SCORE`, `MIN_LEXICAL`) ont été réglés avec `hashing` : leur valeur avec BGE-M3 n'est pas validée. Ne les modifie pas pour « faire monter » le score sans le dire.
- Le jeu de 70 scénarios a déjà servi à régler le prototype : il n'est **pas indépendant**. Le jeu de 150 scénarios figé reste à construire.

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],capture_output=True,text=True).stdout or "PAS DE GPU")
print("CUDA disponible :", torch.cuda.is_available())

In [ ]:
# Dépendances (OCR français pour les 2 PDF scannés + bibliothèques du projet)
!apt-get -qq install -y tesseract-ocr tesseract-ocr-fra > /dev/null
!pip -q install pymupdf qdrant-client pydantic pydantic-settings pytesseract reportlab pillow sentence-transformers pyyaml

In [ ]:
# Récupération du dépôt (public)
import os
if not os.path.exists("MaintiDoc"):
    !git clone -q https://github.com/ziadserray04/MaintiDoc.git
%cd MaintiDoc
!git log --oneline | head -3

In [ ]:
# Régénère le corpus complet (manifest, facts) : déterministe, ~1 minute
!python -m scripts.generate_corpus

In [ ]:
# 1) Référence : prototype avec le backend de test (hashing)
!EMBEDDING_BACKEND=hashing python -m scripts.eval_text_reference docs/eval_hashing.md | tail -25

In [ ]:
# 2) PREMIER PASSAGE avec BGE-M3 (le télécharge : ~2,2 Go, quelques minutes)
!EMBEDDING_BACKEND=bge-m3 python -m scripts.eval_text_reference docs/eval_bge-m3_premier_passage.md | tail -60

In [ ]:
# 3) Jeu de développement (64 questions)
!EMBEDDING_BACKEND=bge-m3 python -m scripts.eval_smoke

In [ ]:
# 4) Comparaison des chiffres clés
import re
def lire(p):
    t=open(p,encoding="utf-8").read()
    g=lambda k: (re.search(rf"\|\s*{re.escape(k)}[^|]*\|\s*([^|]+)\|",t) or [None,"?"])[1].strip()
    tot=re.search(r"Total\*\*\s*\|\s*\*\*(\d+/\d+)",t)
    return {"Recall@5":g("Recall@5"),"nDCG@5":g("nDCG@5"),"MRR@10":g("MRR@10"),"Total":tot[1] if tot else "?"}
import pandas as pd
print(pd.DataFrame({"hashing":lire("docs/eval_hashing.md"),"bge-m3 (1er passage)":lire("docs/eval_bge-m3_premier_passage.md")}))

In [ ]:
# 5) Récupère les rapports pour les envoyer à Claude / les versionner
!zip -q resultats_semaine2.zip docs/eval_hashing.md docs/eval_bge-m3_premier_passage.md
from google.colab import files
files.download("resultats_semaine2.zip")

## Ensuite
Envoie `resultats_semaine2.zip` à Claude. On lira les échecs **avant** de régler quoi que ce soit, et on notera dans le rapport ce qui a changé et pourquoi.